# *<center> Field Export: any device, one drive cycle </center>*

### The following notebook is a general-purpose EXPORTER. It loads any ion_gym geometry, lets you set the drive by hand, and writes the time-resolved electric field over a chosen window to a portable file. Specific topics include:

* Loading a geometry from JSON and reading its electrodes and groups.
* Declaring **per-electrode voltages** and **per-group frequencies**
  interactively — nothing is inherited silently from the file.
* Choosing **solver parameters**: the field method (electrode-aware vs
  plain gradient) and the working precision (float64 vs float32).
* Setting the **time step** and the **total export time**.
* Getting a **file-size estimate before you commit**, then exporting.

This is not tied to any one device: it works for the funnel, the
quadrupole, an einzel lens, a SLIM plane — anything that builds.

### Conventions used in this document:

* **Units are mm, eV, µs, volts, Hz** unless a name says otherwise.
* **CAPITALS are parameters you may change**; the interactive controls
  write into these, and the plain assignment beside each is the fallback
  if you would rather not use the widgets.
* **The JSON supplies GEOMETRY**; voltages, frequencies, precision, time
  step and window are declared here.
* The exported field is the one the tracer flies,
  $\mathbf{E}(\mathbf{x},t) = -\nabla \Phi(\mathbf{x},t)$, where
  $\Phi(\mathbf{x},t) = A(\mathbf{x}) + \sum_k \sin(2\pi f_k t + \varphi_k)\, B_k(\mathbf{x})$ is a
  composition of solved bases, not a pseudopotential.

#### Relevant References

* Ion funnel / RF confinement background as in notebook 03; quadrupole
  drive model as in notebook 02.

____

## Stage 0 — imports

## The instrument, before any statistics

The device this notebook flies by default, drawn from the **solver's own
electrode mask** (not a redrawing) with example ion paths exactly as
flown. You are looking at a tapered stack of RF ring electrodes: the
ring bores shrink toward the exit, adjacent rings carry opposite RF
phases (the effective-potential wall that keeps ions off the metal), and
a DC gradient walks the ions along the axis through buffer gas. The
paths converge radially as the taper narrows — the funnel doing its job:
accepting a wide, diffuse cloud and delivering a thin beam to the exit.

Deck: `examples/ion_funnel_rz.json` (the default — Stage A lets you
point `SPEC_PATH` at any example, and the stages below adapt to whatever
you load). A geometry figure is not decoration — if the picture and the
solved model can disagree, every number below is unverifiable.

In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/ion_funnel_rz.json', banked='panel_funnel.png', height=520)


In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# path anchor (2026-08-08): every relative path below is REPO-ROOT-relative,
# independent of where Jupyter/VS Code set the working directory.
from ion_gym.io.paths import repo_root as _repo_root
import os as _os
_os.chdir(_repo_root())

import os
import numpy as np
from IPython.display import display, Markdown

from ion_gym.io.sim_spec import SimSpec, set_pitch
from ion_gym.physics.sim_build import build_run, sizing_for
from ion_gym.io import field_cycle_io as FC

# ipywidgets is optional: if present you get sliders/dropdowns; if not,
# the plain-variable fallbacks below still make the notebook run.
try:
    import ipywidgets as W
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not installed — using the plain-variable fallbacks "
          "(pip install ipywidgets for the interactive controls).")

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


## Stage A — load a geometry

Point `SPEC_PATH` at any example. The cell reads it and lists what the
file declares: the electrodes, their current DC, and any RF/DC groups.
Nothing here commits you to those values — Stage B is where you set them.

In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# ---- The GEOMETRY source -- change to any example ------------------------
SPEC_PATH = str(ROOT / 'examples/ion_funnel_rz.json')
# others to try:
#   str(ROOT / 'examples/quadrupole_stl_rods_transport.json')
#   str(ROOT / 'examples/einzel_round_r-z.json')
#   str(ROOT / 'examples/slim_tetramer_confinement_2-d_acrossxgap.json')

spec = SimSpec.from_json(SPEC_PATH)
electrodes = list(spec.geometry.electrodes)
rf_groups = list(getattr(spec.geometry, "rf_groups", []) or [])
dc_groups = list(getattr(spec.geometry, "dc_groups", []) or [])

print(f"loaded {spec.name!r}: {len(electrodes)} electrodes, "
      f"{len(rf_groups)} RF group(s), {len(dc_groups)} DC group(s)\n")
rows = ["| electrode | dc (V) | RF group | dc_group |", "|---|---|---|---|"]
for el in electrodes:
    rows.append(f"| {el.name} | {el.dc:g} | "
                f"{', '.join(el.rf_groups) if el.rf_groups else '-'} | "
                f"{el.dc_group or '-'} |")
display(Markdown("\n".join(rows)))
if rf_groups:
    grp = ["| RF group | amplitude (V) | frequency (Hz) | phase (deg) |",
           "|---|---|---|---|"]
    for g in rf_groups:
        grp.append(f"| {g.name} | {g.amplitude_v:g} | {g.frequency_hz:g} | "
                   f"{g.phase_deg:g} |")
    display(Markdown("\n".join(grp)))
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(spec, **DECK_OVERRIDES)


## Stage B — declare voltages and frequencies

Set the DC on each electrode and the amplitude, frequency and phase of
each RF group. The controls below are grouped; if `ipywidgets` is not
installed, edit the `VOLTAGES`, `RF_AMPLITUDE`, `RF_FREQUENCY` and
`RF_PHASE` dictionaries in the fallback cell instead — they are read the
same way downstream.

In [ ]:
# Build the controls (or the fallback dicts) for voltages and RF.
if HAVE_WIDGETS:
    v_widgets = {el.name: W.FloatText(value=float(el.dc), description=el.name,
                                      style={"description_width": "120px"})
                 for el in electrodes}
    rf_amp_w = {g.name: W.FloatText(value=float(g.amplitude_v),
                                    description=f"{g.name} amp (V)",
                                    style={"description_width": "120px"})
                for g in rf_groups}
    rf_freq_w = {g.name: W.FloatText(value=float(g.frequency_hz),
                                     description=f"{g.name} freq (Hz)",
                                     style={"description_width": "120px"})
                 for g in rf_groups}
    rf_phase_w = {g.name: W.FloatText(value=float(g.phase_deg),
                                      description=f"{g.name} phase (deg)",
                                      style={"description_width": "120px"})
                  for g in rf_groups}
    boxes = [W.HTML("<b>Electrode DC voltages</b>")]
    boxes += list(v_widgets.values())
    if rf_groups:
        boxes.append(W.HTML("<b>RF groups</b>"))
        for g in rf_groups:
            boxes += [rf_amp_w[g.name], rf_freq_w[g.name], rf_phase_w[g.name]]
    display(W.VBox(boxes))
else:
    print("Edit these dicts (fallback, no widgets):")
    VOLTAGES = {el.name: float(el.dc) for el in electrodes}
    RF_AMPLITUDE = {g.name: float(g.amplitude_v) for g in rf_groups}
    RF_FREQUENCY = {g.name: float(g.frequency_hz) for g in rf_groups}
    RF_PHASE = {g.name: float(g.phase_deg) for g in rf_groups}
    print("VOLTAGES =", VOLTAGES)
    print("RF_AMPLITUDE =", RF_AMPLITUDE)
    print("RF_FREQUENCY =", RF_FREQUENCY)
    print("RF_PHASE =", RF_PHASE)

## Stage C — solver parameters

Two choices affect the solved field itself:

* **Field method.** `electrode_aware` takes one-sided differences at
  metal-adjacent nodes, so the field right at an electrode surface is
  correct; a `plain_gradient` uses central differences everywhere, which
  halves the surface field but is marginally cheaper. For export you almost
  always want `electrode_aware` — the surface field is usually the point.
* **Working precision.** `float64` solves and stores at full precision;
  `float32` halves the memory (and the exported file) at ~7 significant
  digits, which is ample for display and most transport work.
* **Grid pitch** (`mm_per_gu`) is the size of one solve cell. A finer
  pitch resolves the field more sharply and near electrode surfaces, but
  the node count — and so both the solve time and the exported file —
  grows as (1/pitch) per axis: halving the pitch roughly **quadruples** a
  2-D grid and **octuples** a 3-D one. The default is the value in the
  file; the estimate in Stage E updates the moment you change it, so you
  can trade resolution against size before committing.

**The pitch is not independently settable, and Stage E handles that for
you.** Domain extents are *lattice* quantities: the width of the box is
a whole number of cells times the pitch, exactly, on every axis and
route (charter A7). Type a pitch that does not divide the box the deck
declares and the extent is no longer an integer count, so the loader
refuses the spec before it ever solves — e.g. a 37.05 mm domain at
0.1 mm/cell is 370.5 cells, which is not a domain. This is not
pedantry: lattice errors are *systematic*, they displace symmetry
planes, and unlike rasterization error they do not shrink as you
refine.

Stage E therefore calls `set_pitch(spec, mm_per_gu)` rather than
assigning `mm_per_gu` by hand. It re-counts the domain at your pitch
and absorbs the remainder **at the outer walls**, growing the vacuum
box by less than one cell. Three things follow, and they are the whole
reason it is safe to let a notebook do this automatically:

* **Electrode metal never moves.** Not snapped, not rounded, not
  re-centred — a 0.55 mm gap is still 0.55 mm at every pitch. Metal
  edges stay in mm and rasterize; that is the measured-hardware
  carve-out, and it is why this snap cannot quietly change your device.
* **The box grows outward, never inward.** Trimming to the nearest
  smaller extent would put a wall *inside* the metal it was sized to
  contain.
* **Which nodes land inside the metal does change.** That *is* the
  resolution change, and it is the point of asking for one.

Stage E prints every wall that moved, with its new cell count, so the
domain you solved is never different from the domain you think you
solved. On one case the call refuses instead: a 2-D axis that folds
about the *implicit* midline with no declared `plane_mm`, where growing
a wall would drift the fold plane off the metal it mirrors. The refusal
names the two ways out; declaring the plane is usually the right one.

In [ ]:
if HAVE_WIDGETS:
    field_method_w = W.Dropdown(
        options=["electrode_aware", "plain_gradient"], value="electrode_aware",
        description="field method", style={"description_width": "120px"})
    dtype_w = W.Dropdown(
        options=["float64", "float32"], value="float32",
        description="precision", style={"description_width": "120px"})
    pitch_w = W.FloatText(
        value=float(spec.geometry.mm_per_gu),
        description="grid pitch (mm)", style={"description_width": "120px"})
    display(W.VBox([field_method_w, dtype_w, pitch_w]))
else:
    FIELD_METHOD = "electrode_aware"   # or "plain_gradient"
    CHANNEL_DTYPE = "float32"          # or "float64"
    GRID_PITCH_MM = float(spec.geometry.mm_per_gu)   # smaller = finer grid
    print(f"FIELD_METHOD = {FIELD_METHOD!r}; CHANNEL_DTYPE = {CHANNEL_DTYPE!r}; "
          f"GRID_PITCH_MM = {GRID_PITCH_MM}")

## Stage D — time step and total export time

The **time step** is the interval between exported field frames. The
**total export time** is the window: one drive period reproduces a single
cycle (the usual request); a multiple captures several. The cell derives
the drive period from the frequencies you set in Stage B and offers one
period as the default window, but you can set any window you like.

Number of frames is `total_time / time_step`, and that count multiplies
the file size directly — which is why the estimate in Stage E depends on
it.

In [ ]:
# Read the drive frequency the window defaults to (first RF group, if any).
_first_freq = (float(rf_freq_w[rf_groups[0].name].value)
               if (HAVE_WIDGETS and rf_groups)
               else (RF_FREQUENCY[rf_groups[0].name]
                     if (not HAVE_WIDGETS and rf_groups) else 0.0))
_period_us = (1e6 / _first_freq) if _first_freq > 0 else 1.0

if HAVE_WIDGETS:
    dt_w = W.FloatText(value=round(_period_us / 64, 6),
                       description="time step (µs)",
                       style={"description_width": "120px"})
    ttime_w = W.FloatText(value=round(_period_us, 6),
                          description="total time (µs)",
                          style={"description_width": "120px"})
    display(W.VBox([
        W.HTML(f"drive period ≈ <b>{_period_us:.4g} µs</b> "
               f"({_first_freq/1e3:.1f} kHz)"),
        dt_w, ttime_w]))
else:
    TIME_STEP_US = round(_period_us / 64, 6)   # 64 frames per period
    TOTAL_TIME_US = round(_period_us, 6)       # one period
    print(f"drive period ≈ {_period_us:.4g} µs")
    print(f"TIME_STEP_US = {TIME_STEP_US}; TOTAL_TIME_US = {TOTAL_TIME_US}")

## Stage E — apply the declarations, build, and estimate the file size

This cell reads every control (or fallback), writes the values into the
spec, chooses the solver parameters, **builds the field** (a solve on the
first run for this geometry, a cache hit after), and then prints an
estimated export size — **before** anything is written to disk. The
estimate is the uncompressed array size, so it is a safe upper bound; npz
compression on smooth fields usually lands well under it.

In [ ]:
def read_controls():
    'Collect the declarations from widgets or fallbacks into one dict.'
    if HAVE_WIDGETS:
        volts = {n: float(w.value) for n, w in v_widgets.items()}
        amp = {n: float(w.value) for n, w in rf_amp_w.items()}
        freq = {n: float(w.value) for n, w in rf_freq_w.items()}
        phase = {n: float(w.value) for n, w in rf_phase_w.items()}
        method = field_method_w.value
        dtype = dtype_w.value
        pitch = float(pitch_w.value)
        dt_us = float(dt_w.value)
        total_us = float(ttime_w.value)
    else:
        volts, amp, freq, phase = (VOLTAGES, RF_AMPLITUDE,
                                   RF_FREQUENCY, RF_PHASE)
        method, dtype = FIELD_METHOD, CHANNEL_DTYPE
        pitch = GRID_PITCH_MM
        dt_us, total_us = TIME_STEP_US, TOTAL_TIME_US
    if dt_us <= 0 or total_us <= 0:
        raise ValueError("time step and total time must both be positive")
    if pitch <= 0:
        raise ValueError("grid pitch must be positive")
    return dict(volts=volts, amp=amp, freq=freq, phase=phase,
                method=method, dtype=dtype, pitch=pitch,
                dt_us=dt_us, total_us=total_us)

ctrl = read_controls()

# Apply the operating point onto the spec (geometry stays from the file).
export_spec = SimSpec.from_json(SPEC_PATH)
for el in export_spec.geometry.electrodes:
    if el.name in ctrl["volts"]:
        el.dc = ctrl["volts"][el.name]
for g in export_spec.geometry.rf_groups:
    if g.name in ctrl["amp"]:
        g.amplitude_v = ctrl["amp"][g.name]
        g.frequency_hz = ctrl["freq"][g.name]
        g.phase_deg = ctrl["phase"][g.name]
export_spec.geometry.field_method = ctrl["method"]
export_spec.geometry.channel_dtype = ctrl["dtype"]
# THE PITCH IS NOT INDEPENDENTLY SETTABLE. Domain extents are lattice
# quantities counted in cells (A7), so a pitch that does not divide the
# box the deck declares leaves the spec refusable -- assigning
# mm_per_gu alone is how this cell used to fail the moment anyone typed
# a coarser number (measured: the 3-D SLIM tetramer deck at 0.1 mm/cell
# is 370.5 x 123.5 cells, refused by the loader before any solve).
# set_pitch re-counts the domain and covers the remainder at the OUTER
# WALLS, by less than one cell. Electrode metal never moves.
_pitch_changes = set_pitch(export_spec, ctrl["pitch"])

# WHAT MOVED IS REPORTED, INCLUDING WHEN NOTHING DID -- a domain that
# silently differs from the one you think you solved is the whole
# failure mode this replaces.
_moved = [c for c in _pitch_changes if c.moved()]
if _moved:
    print(f"domain re-counted at {ctrl['pitch']:g} mm/cell "
          f"(metal unchanged; slop added at the outer walls):")
    for _c in _moved:
        print(f"    {_c.describe()}")
else:
    print(f"domain already conforms at {ctrl['pitch']:g} mm/cell "
          f"— no wall moved")

if getattr(export_spec.geometry, "dc_groups", None):
    export_spec.resolve_dc_groups()

# Build (solve on first run, cache hit after).
model, _fly, _cols, _births = build_run(export_spec)

n_frames = max(1, int(round(ctrl["total_us"] / ctrl["dt_us"])))
est = FC.estimate_cycle_bytes(model, n_samples=n_frames, dtype=ctrl["dtype"])
drives = FC.model_drives(model)["drives"]

print(f"device        : {export_spec.name}")
print(f"grid          : {est['grid']}  ({est['n_nodes']:,} nodes)")
print(f"drives        : "
      + (", ".join(f"{n} @ {f/1e3:g} kHz" for n, f, _ in drives)
         if drives else "none (static field)"))
print(f"frames        : {n_frames}  (total {ctrl['total_us']:g} µs / "
      f"step {ctrl['dt_us']:g} µs)")
print(f"precision     : {ctrl['dtype']}   field method: {ctrl['method']}")
print(f"grid pitch    : {ctrl['pitch']:g} mm/cell")
_g = export_spec.geometry
print(f"domain        : "
      + " x ".join(f"{_e:g} mm" for _e in (_g.width_mm, _g.height_mm,
                                           _g.depth_mm) if _e)
      + (f"   ({len(_moved)} wall(s) moved by the pitch snap)"
         if _moved else "   (as declared in the deck)"))
print(f"\nESTIMATED FILE SIZE (upper bound): "
      f"{est['total_bytes']/1e6:.1f} MB")
print(f"  = {n_frames} frames x {est['n_nodes']:,} nodes x 3 components "
      f"x {4 if ctrl['dtype']=='float32' else 8} bytes")
if est['total_bytes'] > 500e6:
    print("  ** over 500 MB — consider float32, fewer frames, or a shorter "
          "window before exporting. **")
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(export_spec, **DECK_OVERRIDES)


## Stage F — export

If the estimate looks right, write the file. The result is one
self-describing `.npz`:

* `t_us` — frame times (µs), spanning your window.
* `Ex, Ey, Ez` — the field components (V/mm); `Ez` is zero for a 2-D
  solve, present so every export has the same shape.
* `x_mm, y_mm, z_mm` — the axis coordinates.
* `drive_freq_hz, drive_phase_rad` — the drive table actually used.
* a JSON `_meta` member — format, window, grid, units, drives, precision.

Read it back anywhere with `FC.read_field_cycle(path)`.

In [ ]:
# ---- The output file -- change to suit your system -----------------------
# outputs go in notebooks/out/, never the repo root
from ion_gym.io import paths as _paths
OUT_PATH = str(_paths.outputs_dir(
    f"exported_field_{export_spec.name}.npz".replace(" ", "_")))

path, meta = FC.save_model_cycle(
    model, OUT_PATH,
    n_samples=n_frames, total_time_us=ctrl["total_us"],
    dtype=ctrl["dtype"], label=f"{export_spec.name} field export")
actual_mb = os.path.getsize(path) / 1e6
print(f"wrote {os.path.abspath(path)}  ({actual_mb:.1f} MB on disk, "
      f"npz-compressed from the {est['total_bytes']/1e6:.1f} MB estimate)")

# Prove it round-trips and holds the field the tracer would fly.
data, md = FC.read_field_cycle(path)
print(f"\nread back: Ex {data['Ex'].shape} {data['Ex'].dtype}, "
      f"t_us {data['t_us'].shape} spanning "
      f"{data['t_us'][0]:.4g}..{data['t_us'][-1]:.4g} µs")
print(f"|E| range over the window: {np.hypot(data['Ex'], data['Ey']).min():.2f}"
      f" .. {np.hypot(data['Ex'], data['Ey']).max():.2f} V/mm")
print(f"meta: {', '.join(f'{k}={md[k]}' for k in ('format','compose','period_us','dtype'))}")

### A quick look at what was exported

A montage of $|\\mathbf{E}|$ at a few frames across the window, straight
from the file just written — a sanity check that the field varies over the
cycle the way the drive says it should.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data, md = FC.read_field_cycle(OUT_PATH)
# |E| including Ez (zero for a 2-D solve, real for 3-D).
Emag = np.sqrt(data["Ex"]**2 + data["Ey"]**2 + data["Ez"]**2)
# For a 3-D field, show a mid-plane slice so the montage is 2-D panels;
# for a 2-D field this is a no-op.
if Emag.ndim == 4:                    # (nt, nx, ny, nz) -> mid-z slice
    kz = Emag.shape[3] // 2
    Emag = Emag[:, :, :, kz]
    print(f"3-D field: montage shows the mid-plane slice z = "
          f"{data['z_mm'][kz]:.2f} mm (the file holds the full volume).")
n_show = min(4, Emag.shape[0])
idx = np.linspace(0, Emag.shape[0] - 1, n_show).astype(int)
fig = make_subplots(rows=1, cols=n_show,
                    subplot_titles=[f"t = {data['t_us'][i]:.3g} µs"
                                    for i in idx])
zmax = float(np.percentile(Emag, 99))
for c, i in enumerate(idx, start=1):
    fig.add_heatmap(z=Emag[i].T, x=data["x_mm"], y=data["y_mm"],
                    zmin=0, zmax=zmax, colorscale="Magma",
                    showscale=(c == n_show),
                    colorbar=dict(title="|E| (V/mm)") if c == n_show else None,
                    row=1, col=c)
    fig.update_xaxes(title_text="x (mm)", row=1, col=c)
fig.update_yaxes(title_text="y (mm)", row=1, col=1)
fig.update_layout(height=360, width=280 * n_show,
                  title_text=f"{md.get('label','')} — |E| across the window",
                  margin=dict(l=60, r=20, t=60, b=50))
fig

____

## Reading the file elsewhere (no ion_gym required)

The export is a plain NumPy `.npz`. Anything that can read one — NumPy in
any environment, or `scipy`, MATLAB, Julia's NPZ.jl — can open it; **you
do not need ion_gym installed to read it.** The only ion_gym-specific
detail is that the metadata is stored as a JSON string packed into a
`uint8` array named `_meta` (npz members must be arrays), so you decode
those bytes back to text and parse them.

### File layout

| member | shape | dtype | meaning |
|---|---|---|---|
| `t_us` | `(nt,)` | float64 | frame times, µs |
| `Ex`, `Ey`, `Ez` | `(nt, nx, ny)` | float32/64 | field components, V/mm |
| `x_mm`, `y_mm`, `z_mm` | `(nx,)`,`(ny,)`,`(nz,)` | float64 | axis coordinates, mm |
| `drive_freq_hz` | `(n_drives,)` | float64 | drive frequencies |
| `drive_phase_rad` | `(n_drives,)` | float64 | drive phases |
| `_meta` | `(?,)` | uint8 | UTF-8 JSON: format, grid, units, drives, dtype, label |

`Ex[k]` is the x-component of the field over the whole grid at time
`t_us[k]`; the value at grid node `(i, j)` sits at physical position
`(x_mm[i], y_mm[j])`. For a 2-D solve `Ez` is all zeros and `z_mm` has one
entry — the plane-normal field is zero in the solved plane.

### A complete, dependency-free reader

Copy this into any Python with NumPy; it needs nothing from ion_gym.

In [ ]:
# ---- standalone reader: numpy + json only, no ion_gym --------------------
import numpy as np
import json

def read_exported_field(path):
    """Read an ion_gym field-cycle .npz. Requires only numpy + json.

    Returns (arrays, meta):
      arrays : dict of numpy arrays (t_us, Ex, Ey, Ez, x_mm, y_mm, z_mm, ...)
      meta   : plain dict decoded from the JSON in the _meta member
    """
    npz = np.load(path, allow_pickle=False)
    meta = {}
    if "_meta" in npz.files:
        meta = json.loads(bytes(npz["_meta"]).decode("utf-8"))
    arrays = {k: npz[k] for k in npz.files if k != "_meta"}
    return arrays, meta


# --- example use -----------------------------------------------------------
arrays, meta = read_exported_field(OUT_PATH)     # any .npz this notebook wrote
Ex, Ey, Ez = arrays["Ex"], arrays["Ey"], arrays["Ez"]  # (nt, nx, ny), V/mm
t_us       = arrays["t_us"]                            # (nt,)
x_mm       = arrays["x_mm"]                            # (nx,)
y_mm       = arrays["y_mm"]                            # (ny,)

# |E| everywhere at the first frame:
Emag0 = np.sqrt(Ex[0]**2 + Ey[0]**2 + Ez[0]**2)

# the field vector at grid node (i, j), sitting at (x_mm[i], y_mm[j]):
i, j = 10, 20
Ex_ij_over_time = Ex[:, i, j]                          # (nt,)

# what device / drive produced this:
print("device :", meta.get("label"))
print("drives :", meta.get("drives"))
print("period :", meta.get("period_us"), "us")
print("grid   :", meta.get("grid"))
print()
print("verified on this notebook's own export:", Ex.shape, Ex.dtype,
      "spanning", round(float(t_us[0]), 4), "to",
      round(float(t_us[-1]), 4), "us")

### Reading it in other languages

The same file opens without Python at all:

* **MATLAB** — an `.npz` is a zip of `.npy` files. Unzip it, then read
  each array with a `readNPY` helper (e.g. the widely-used `npy-matlab`);
  the `_meta` member is a byte vector you convert with `char()` and parse
  with `jsondecode()`.
* **Julia** — `using NPZ; d = npzread("export.npz")` returns a `Dict`;
  `String(UInt8.(d["_meta"]))` then `JSON.parse` for the metadata.
* **Raw** — because it is just a zip, `unzip export.npz` gives you
  `Ex.npy`, `t_us.npy`, ... which any NPY reader loads directly.

____

### Notes

* **SLIM and other channel-pack devices.** This notebook composes from a
  built `SimSpec` model (`A + Σ B_k`). The SLIM tetramer builds its field
  as an explicit channel pack instead; for that device, use
  `FC.export_slim_rf_cycle(path, tw_amp_v=0.0)` (or `FC.save_field_cycle`
  on the pack), which lets you zero the travelling wave. Same file format,
  same reader.
* **Static devices.** If the geometry has no RF group, there is no cycle;
  pass `total_time_us` to export a fixed window (every frame is identical,
  and the metadata says `compose: static`).
* **Precision.** `float32` is the right default for display and most
  transport; switch to `float64` when you need the field at full solver
  precision (e.g. feeding another integrator).

## Read-out

- **One drive cycle is the complete description of a periodic field.** Because the solver stores a static basis plus one basis per drive, any instant of the waveform is a weighted sum — so exporting a cycle exports everything downstream code needs, without re-solving.
- **What the export must carry to be reusable:** the grid pitch and origin, the drive frequencies and phases, and the geometry hash that produced it. An exported field without its provenance cannot be checked against the model that made it, and silently stale fields are the classic way a simulation lies.